In [1]:
!pip install copernicusmarine
!pip install netCDF4
!pip install seaborn
!pip install scipy

In [2]:
import copernicusmarine
from netCDF4 import Dataset, num2date
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import os
import xarray as xr
import seaborn as sns
import seaborn.objects as so
import numpy as np
import gc
from datetime import datetime
import glob
from scipy import stats

# Copernicus Data

Currently Contains:
- Kd490 Data, Statistics, and Maps
- Chlorophyll Statistics and Maps
- CDOM Data, Statistics, and Maps
- SST Data, Statistics, and Maps
- SPM Data, Statistics, and Maps

Analysis of trends done in Analysis.ipynb

Notes Taken While Creating: https://nuigalwayie-my.sharepoint.com/:w:/g/personal/h_doran1_universityofgalway_ie/EZ9BR9QbsnBDkOLLm5LP284BbncLs8fV3ScHW9HXyz42Jw?e=2Loh5a


Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).


*Only need to run the cell below once per computer, unless adding a new library*

*Run cell below everytime open file*

# First - Kd490!

### wrong file function, move to the next

In [ ]:
## sets username and password so I don't have to input it everytime

os.environ['COPERNICUS_USERNAME'] = 'hdoran'
os.environ['COPERNICUS_PASSWORD'] = "ihateAngela69**##"

In [ ]:
## this is the wrong function, the next one (new_batch_file) is what was run (wanted files in different directory)
username = os.getenv('COPERNICUS_USERNAME')
password = os.getenv('COPERNICUS_PASSWORD')

def batch_file(dataset_id, variables, regions):
    start_date = datetime(1997, 9, 4)
    end_date = datetime(2025, 9, 4)

    results = {}
    years = range(start_date.year, end_date.year + 1)

    for region in regions:
        region_name = region["name"]
        region_file_paths = []

        for year in years:
            year_start = datetime(year, 1, 1)
            year_end = datetime(year, 12, 31)
            if year == start_date.year:
                year_start = start_date
            if year == end_date.year:
                year_end = end_date

            # Download the data for this specific year
            result = copernicusmarine.subset(
                dataset_id=dataset_id,
                variables=variables,
                minimum_longitude=region["min_lon"],
                maximum_longitude=region["max_lon"],
                minimum_latitude=region["min_lat"],
                maximum_latitude=region["max_lat"],
                start_datetime=year_start.strftime('%Y-%m-%dT%H:%M:%S'),
                end_datetime=year_end.strftime('%Y-%m-%dT%H:%M:%S'),
                output_directory='/home/jovyan/my_new_directory',
                username=username,
                password=password
            )

            # Save only the file path (not the dataset object)
            region_file_paths.append(result.file_path)

            # Free memory
            del result
            gc.collect()

        results[region_name] = region_file_paths

    return results


In [ ]:
ireland_file = batch_file("cmems_obs-oc_atl_bgc-transp_my_l3-multi-1km_P1D",['KD490'],[{"name":"ireland", "min_lon": -13.0, "min_lat": 50.0, "max_lon":-5.0, "max_lat": 55.0}])

In [ ]:
ds = xr.open_dataset('/home/jovyan/my_new_directory/cmems_obs-oc_atl_bgc-transp_my_l3-multi-1km_P1D_KD490_12.99W-5.01W_50.01N-54.99N_2023-01-01-2023-12-31.nc')

In [ ]:
ds

In [ ]:
ds['KD490'].isel(time=0).plot()

### Correct File Function and Extraction - Use as template for other variables

In [ ]:
os.environ['COPERNICUS_USERNAME'] = 'hdoran'
os.environ['COPERNICUS_PASSWORD'] = "ihateAngela69**##"

username = os.getenv('COPERNICUS_USERNAME')
password = os.getenv('COPERNICUS_PASSWORD')
## messed up! should've done at least 56 degrees north, or maybe even 57
## 55 deg N cuts into the northern end of the island
def new_batch_file(dataset_id, variables, regions):
    start_date = datetime(1997, 9, 4)
    end_date = datetime(2025, 9, 4)

    results = {}
    years = range(start_date.year, end_date.year + 1)

    for region in regions:
        region_name = region["name"]
        region_file_paths = []

        for year in years:
            year_start = datetime(year, 1, 1)
            year_end = datetime(year, 12, 31)
            if year == start_date.year:
                year_start = start_date
            if year == end_date.year:
                year_end = end_date

            # Download the data for this specific year
            result = copernicusmarine.subset(
                dataset_id=dataset_id,
                variables=variables,
                minimum_longitude=region["min_lon"],
                maximum_longitude=region["max_lon"],
                minimum_latitude=region["min_lat"],
                maximum_latitude=region["max_lat"],
                start_datetime=year_start.strftime('%Y-%m-%dT%H:%M:%S'),
                end_datetime=year_end.strftime('%Y-%m-%dT%H:%M:%S'),
                output_directory='/home/jovyan/notmessedup_directory',
                username=username,
                password=password
            )

            # Save only the file path (not the dataset object)
            region_file_paths.append(result.file_path)

            # Free memory
            del result
            gc.collect()

        results[region_name] = region_file_paths

    return results


In [ ]:
## Source: North Atlantic Ocean Colour Plankton, Reflectance, Transparency and Optics MY L3 daily observations
## DOI (product):
## https://doi.org/10.48670/moi-00286
## Accessed: October 10, 2025

In [ ]:
big_ireland_file = new_batch_file("cmems_obs-oc_atl_bgc-transp_my_l3-multi-1km_P1D",['KD490'],[{"name":"ireland", "min_lon": -15.0, "min_lat": 48.0, "max_lon":-2.0, "max_lat": 59.0}])

## We have the data! don't run the cells above again. Now, start loadiing the .nc files into the notebook. [This section is more exploratory].

In [ ]:
ds_1997 = xr.open_dataset('/home/jovyan/notmessedup_directory/cmems_obs-oc_atl_bgc-transp_my_l3-multi-1km_P1D_KD490_14.99W-2.01W_48.01N-58.99N_1997-09-04-1997-12-31.nc')

In [ ]:
ds_1997

In [ ]:
ds_1997['KD490'].attrs

In [ ]:
ds_1997['KD490'].attrs['long_name']


In [ ]:
ds_1997['KD490']

In [ ]:
kd490_97 = ds_1997['KD490']
nonmissingcount = kd490_97.notnull().sum(dim=['latitude','longitude'])
total_points = kd490_97.sizes['latitude'] * kd490_97.sizes['longitude']
coverage_fraction = nonmissingcount/total_points
good_days = coverage_fraction >= .10
days_with_15pct = kd490_97['time'][good_days]
days_with_15pct.values

In [ ]:
good_days_97_np = days_with_15pct.to_numpy()

In [ ]:
ds_1997['KD490'].sel(time='1997-10-31').plot()

In [ ]:
ds_1997['KD490'].isel(time= 21).plot()

In [ ]:
for day in good_days_97_np:
    ds_1997['KD490'].sel(time=day).plot()
    plt.title(f"KD490 on {day}")
    plt.show()

In [ ]:
ncols = 5
nrows = int(np.ceil(len(good_days_97_np)/ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows))
axes = axes.flatten()

for i, day in enumerate(good_days_97_np):
    ds_1997['KD490'].sel(time=day).plot(ax=axes[i])
    axes[i].set_title(str(day)[:10], fontsize=10)

# Turn off unused axes
for ax in axes[len(good_days_97_np):]:
  ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
ds_1998 = xr.open_dataset('/home/jovyan/notmessedup_directory/cmems_obs-oc_atl_bgc-transp_my_l3-multi-1km_P1D_KD490_14.99W-2.01W_48.01N-58.99N_1998-01-01-1998-12-31.nc')

In [ ]:
kd490_98 = ds_1998['KD490']
nonmissingcount_98 = kd490_98.notnull().sum(dim=['latitude','longitude'])
total_points_98 = kd490_98.sizes['latitude'] * kd490_98.sizes['longitude']
coverage_fraction_98 = nonmissingcount_98/total_points_98
good_days_98 = coverage_fraction_98 >= .10
days_with_15pct_98 = kd490_98['time'][good_days_98]
good_days_98_np = days_with_15pct_98.to_numpy()

In [ ]:
ncols = 5
nrows = int(np.ceil(len(good_days_98_np)/ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows))
axes = axes.flatten()

for i, day in enumerate(good_days_98_np):
    ds_1998['KD490'].sel(time=day).plot(ax=axes[i])
    axes[i].set_title(str(day)[:10], fontsize=10)

# Turn off unused axes
for ax in axes[len(good_days_98_np):]:
  ax.axis('off')

plt.tight_layout()
plt.show()

## Combine netCDF files into one large dataset

In general, don't need to rerun once the weekly datasets are made and saved to netCDF files in the project file folder

Use as template for other variables

In [ ]:
file_list = sorted(glob.glob('/home/jovyan/notmessedup_directory/*.nc'))
all_ds = xr.open_mfdataset(file_list, concat_dim='time', combine='nested')
## combine all the files into one, massive DataSet so that we can do comparisons over years
all_ds

## Weekly - put into smaller subsets to reduce RAM

In [ ]:
### this is too much RAM - need to do yearly or seasonal analysis
#trend = all_ds['KD490'].polyfit(dim='time', deg=1)
#slope = trend['polyfit_coefficients'].sel(degree=1)
#slope.to_netcdf('KD490_trend.nc')


In [ ]:
# Daily data is already in the dataset
#daily = all_ds['KD490']

# Yearly average
#yearly = all_ds['KD490'].groupby('time.year').mean(dim='time')

# Seasonal average
#seasonal = all_ds['KD490'].groupby('time.season').mean(dim='time')


##decided later that yearly or seasonal was too temporally far apart


In [ ]:
weekly = all_ds['KD490'].resample(time='1W').mean()
weekly
## this cell resamples all the daily data
## gets the mean for each week, cuts the size down from 54 GB to 8 GB
## if no data for the pixel, the mean stays Nan for the weeks pixel

In [ ]:
monthly_kd490 = all_ds['KD490'].resample(time='1ME').mean()
seasonal_kd490 = all_ds['KD490'].resample(time='QS-DEC').mean()
seasonal_kd490['season'] = seasonal_kd490['time'].dt.season
seasonal_kd490

## Polyfit function - fine but not statistically significant, only slope and intercept
Don't need to run this function again - don't do this for other analyses. use stats.linregress

In [ ]:
#time_years = weekly['time'].dt.year + (weekly['time'].dt.dayofyear - 1) / 365.25
t = weekly['time']
time_years = t.dt.year + (t - pd.to_datetime(t.dt.year.astype(str) + '-01-01')) / (pd.to_datetime((t.dt.year + 1).astype(str) + '-01-01') - pd.to_datetime(t.dt.year.astype(str) + '-01-01'))


## time_years = weekly['time'].dt.year + (weekly['time'].dt.dayofyear - 1) / 365.25
## the comment-ed time_years was too approximate
## need to rerun code here to get more accurate time
x = time_years.values.astype(float)

x = time_years.values.astype(float)
## the above line turns the years given into fractional years
##  eg. 2002.765, 2002.770 or something like that
## x is the fractional years, and is used for the regression
## Slope (of KD490) computed later is now in units of m^-1 yr^-1, instead of just m^-1


def _fit_coeffs(y, x=x, min_points=20):
    ## y is weekly time series at one spatial location (in this case, one 1km*1km pixel)
    mask = np.isfinite(y)
    ## the mask (np.isfinite(y)) identifies valid weeks for the pixel in question (only uses non_NAN values
    if mask.sum() < min_points:
        return np.nan, np.nan
    ## if there are less than (in this case) 20 valid weeks of data, then it returns Nan for both slope and intercept.
    p = np.polyfit(x[mask], y[mask], 1)
    ## np.polyfit returns a 1st degree polynomial (straight line) to x (time) and y (weekly KD490 means) with ordinary least squares.
    return p[0], p[1]  # slope per year, intercept

## np.polyfit does not return uncertainty, p-values or R^2. only coefficients




In [ ]:
slope, intercept = xr.apply_ufunc(
    _fit_coeffs,
    weekly,
    input_core_dims=[['time']],
    ## applys _fit_coeffs along the time dimension for each lat long
    output_core_dims=[[], []],
    ## outputs two scalar outputs for each input (pixel)
    vectorize=True,
    ## vectorize allows function to be broadcast over the non core dimensions (lat long)
    dask='parallelized',
    ## uses dask arrays to run
    dask_gufunc_kwargs={'allow_rechunk': True},
    ## if its needs to, it can rechunk
    output_dtypes=[float, float],
    ## the dtype of outputs
)
## xr.apply_ufunc() allows you to apply _fit_coeffs to the whole dataset
## slope, intercept are the variable names for the DataArrays generated by the function. Has latitude and longitude as dimensions

slope.name = 'KD490_weekly_slope_per_year'
intercept.name = 'KD490_weekly_intercept'
## this just sets the name attribute for the netCDF files. More descriptive.

In [ ]:
## this cell pertains to the percent change

years_span = x.max() - x.min()   # ~29 years
mean_val = weekly.mean(dim='time')



percent_change = (slope * years_span) / mean_val * 100
## for each pixel, the slope is multiplied by x and then divided by the mean y times 100 to get the percent change.
## slope * year_span is the absolute change in KD490 for the pixels
## divide by mean val to get dimensionless change as fraction of the mean
percent_change.name = 'KD490_percent_change_29yr'


***Don't need to run the cell below again. Already saved to netcdf file***

In [ ]:
ds_to_save = xr.Dataset({
    'slope': slope,
    'intercept': intercept,
    'percent_change': percent_change
})
## new dataset with the dataarrays of interest, slope, intercept, and percent change

encoding = {v: {"zlib": True, "complevel": 4} for v in ds_to_save.data_vars}
## this line above is to reduce file size

ds_to_save.to_netcdf('KD490_weekly_trend.nc', encoding=encoding, compute=True)
## saves at netcdf

In [ ]:
os.getcwd()
## location of where the file is saved to on computer

In [ ]:
ds_to_save

### Open datasets that are already run

In [ ]:
trend_ds = xr.open_dataset('thesis_data/KD490_weekly_trend.nc')
trend_ds

In [ ]:
plt.figure(figsize=(12, 8))
trend_ds['slope'].plot(
    cmap='RdBu_r',      # red = darkening, blue = brightening
    vmin=-0.002, vmax=0.002  # adjust to your data range
)
plt.title('KD490 Weekly Trend Slope')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

In [ ]:
trend_ds['slope'].min()

In [ ]:
#https://www.youtube.com/watch?v=0YsFR6xqic8
#https://ethan-campbell.github.io/OCEAN_215/calendar/

## Time for statistics! - stats.linregress()


In [ ]:
## use weekly dataarray again

t = weekly['time']
time_years = t.dt.year + (t - pd.to_datetime(t.dt.year.astype(str) + '-01-01')) / (pd.to_datetime((t.dt.year + 1).astype(str) + '-01-01') - pd.to_datetime(t.dt.year.astype(str) + '-01-01'))


## time_years = weekly['time'].dt.year + (weekly['time'].dt.dayofyear - 1) / 365.25
## the comment-ed time_years was too approximate
## need to rerun code here to get more accurate time oct 14 2025 - has been rerun - files are updated oct 17, 2025
x = time_years.values.astype(float)
## repeated to make sure they exist when running this cell

# Run the cell below anytime doing stat analysis!

In [3]:

def fit_with_uncertainty(y, x, min_points=20):
    mask = np.isfinite(y)
    if mask.sum() < min_points:
        # slope, intercept, r, p, stderr_slope, stderr_intercept
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan

    res = stats.linregress(x[mask], y[mask])
    return (
        res.slope,
        res.intercept,
        res.rvalue,
        res.pvalue,
        res.stderr,             # standard error of the slope
        getattr(res, "intercept_stderr", np.nan)  # available in SciPy >=1.9
    )

## back to Stats - linregress

In [ ]:

slope, intercept, rvalue, pvalue, slope_stderr, intercept_stderr = xr.apply_ufunc(
    fit_with_uncertainty,
    weekly,
    input_core_dims=[['time']],
    output_core_dims=[[], [], [], [], [], []],
    vectorize=True,
    dask='parallelized',
    dask_gufunc_kwargs={'allow_rechunk': True},
    output_dtypes=[float, float, float, float, float, float],
    kwargs={'x': x}
)


In [ ]:
stat_trend_ds = xr.Dataset({
    'slope': slope,
    'intercept': intercept,
    'rvalue': rvalue,
    'pvalue': pvalue,
    'slope_stderr': slope_stderr,
    'intercept_stderr': intercept_stderr,
})


In [ ]:
stat_trend_ds = stat_trend_ds.compute()

In [ ]:

stat_trend_ds.to_netcdf('KD490_trend_with_stats.nc')

In [ ]:
stat_trend_ds = xr.open_dataset('thesis_data/KD490_trend_with_stats.nc')

In [ ]:
significant = stat_trend_ds['pvalue'] < 0.05  # 95% confidence
trend_sig = stat_trend_ds['slope'].where(significant)

### Plots

In [ ]:
stat_trend_ds['slope'].plot(
    cmap='RdBu_r',
    center=0,
    vmin=-0.002, vmax=0.002,
    cbar_kwargs={'label': 'KD490 Trend (m^-1 per year)'}
)
plt.title('KD490 Trend per Year')

plt.savefig('Weekly_Trend_KD490.png')
plt.show()


In [ ]:
trend_sig.plot(
    cmap='RdBu_r',
    center=0,
    vmin=-0.002, vmax=0.002,
    cbar_kwargs={'label': 'Significant KD490 Trend (m^-1 per year)'}
)
plt.savefig('Significant_Trend_KD490.png')
plt.show()
## shows the slope of the values where the p-value was less than 0.05

In [ ]:
plt.figure(figsize=(10,6))
stat_trend_ds['pvalue'].plot(
    cmap='viridis_r',  # darker = smaller p-values (more significant)
    vmin=0, vmax=0.1
)
plt.title('Trend Significance (p-value)')
plt.savefig('Trend_pvalue_KD490.png')
plt.show()

In [ ]:
stat_trend_ds['slope']

## Adding Seasons

In [ ]:
# Define a season label for each time step
def assign_season(month):
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    elif month in [9, 10, 11]:
        return 'autumn'

In [ ]:

weekly_season = weekly['time'].dt.month.to_series().apply(assign_season)
weekly = weekly.assign_coords(season=('time', weekly_season.values))


In [ ]:
seasonal_trends = {}

for season in ['winter', 'spring', 'summer', 'autumn']:
    weekly_season_data = weekly.where(weekly['season'] == season, drop=True)

    ## time_years_season = weekly_season_data['time'].dt.year + (weekly_season_data['time'].dt.dayofyear - 1)/365.25
    ## original ^^ is too approximate - need to rerun oct 14 2025 - have rerun oct 17 2025
    t = weekly_season_data['time']
    time_years_season = t.dt.year + (t - pd.to_datetime(t.dt.year.astype(str) + '-01-01')) / (pd.to_datetime((t.dt.year + 1).astype(str) + '-01-01') - pd.to_datetime(t.dt.year.astype(str) + '-01-01'))

    x_season = time_years_season.values.astype(float)

    slope, intercept, rvalue, pvalue, slope_stderr, intercept_stderr = xr.apply_ufunc(
        fit_with_uncertainty,
        weekly_season_data,
        input_core_dims=[['time']],
        output_core_dims=[[], [], [], [], [], []],
        vectorize=True,
        dask='parallelized',
        dask_gufunc_kwargs={'allow_rechunk': True},
        output_dtypes=[float, float, float, float, float, float],
        kwargs={'x': x_season}
    )

    seasonal_trends[season] = xr.Dataset({
        'slope': slope,
        'intercept': intercept,
        'rvalue': rvalue,
        'pvalue': pvalue,
        'slope_stderr': slope_stderr,
        'intercept_stderr': intercept_stderr
    })


In [ ]:
seasonal_trends

In [ ]:
# Add a new coordinate 'season' to each Dataset
season_ds_list = []
for season, ds in seasonal_trends.items():
    ds = ds.expand_dims({'season': [season]})  # adds a season dimension
    season_ds_list.append(ds)

# Concatenate along the new 'season' dimension
all_seasons_ds = xr.concat(season_ds_list, dim='season')

In [ ]:
all_seasons_ds = all_seasons_ds.compute()

In [ ]:

encoding = {v: {"zlib": True, "complevel": 4} for v in all_seasons_ds.data_vars}
all_seasons_ds.to_netcdf('KD490_seasonal_trends.nc', encoding=encoding)

In [ ]:
seasonal_ds = xr.open_dataset('thesis_data/KD490_seasonal_trends.nc')
seasonal_ds

### Seasonal Plots

In [ ]:
w_da = seasonal_ds['slope'].sel(season='winter')


w_da.plot(
    cmap='RdBu_r',
    center=0,
    vmin=-0.002, vmax=0.002,
    cbar_kwargs={'label': 'KD490 Trend (m^-1 per year)'}
)
plt.title('Winter KD490 Trend')
plt.show()

In [ ]:
spr_da = seasonal_ds['slope'].sel(season='spring')


spr_da.plot(
    cmap='RdBu_r',
    center=0,
    vmin=-0.002, vmax=0.002,
    cbar_kwargs={'label': 'KD490 Trend (m^-1 per year)'}
)
plt.title('Spring KD490 Trend')
plt.show()

In [ ]:
sum_da = seasonal_ds['slope'].sel(season='summer')


sum_da.plot(
    cmap='RdBu_r',
    center=0,
    vmin=-0.002, vmax=0.002,
    cbar_kwargs={'label': 'KD490 Trend (m^-1 per year)'}
)
plt.title('Summer KD490 Trend')
plt.show()

In [ ]:
aut_da = seasonal_ds['slope'].sel(season='autumn')


aut_da.plot(
    cmap='RdBu_r',
    center=0,
    vmin=-0.002, vmax=0.002,
    cbar_kwargs={'label': 'KD490 Trend (m^-1 per year)'}
)
plt.title('Autumn KD490 Trend')
plt.show()

import matplotlib.pyplot as plt

seasons = ['winter', 'spring', 'summer', 'autumn']

for season in seasons:
    da = all_seasons_ds['slope'].sel(season=season)

    da.plot(
        cmap='RdBu_r',
        center=0,
        vmin=-0.002,
        vmax=0.002,
        cbar_kwargs={'label': 'KD490 Trend (m^-1 per year)'}
    )

    plt.title(f'{season.capitalize()} KD490 Trend')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.show()



In [ ]:
seasons = [ 'autumn','winter', 'spring', 'summer']
fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)

for ax, season in zip(axes.flat, seasons):
    all_seasons_ds['slope'].sel(season=season).plot(
        ax=ax, cmap='RdBu_r', center=0, vmin=-0.002, vmax=0.002, add_colorbar=False
    )
    ax.set_title(f'{season.capitalize()} KD490 Trend')

# shared colorbar
fig.colorbar(
    plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=-0.002, vmax=0.002)),
    ax=axes.ravel().tolist(), orientation='vertical', fraction=0.03, pad=0.04
)

fig.savefig('Seasonal_KD490.png')
plt.show()

In [ ]:
seasons = ['winter', 'spring', 'summer', 'autumn']
fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

for ax, season in zip(axes.flat, seasons):
    all_seasons_ds['pvalue'].sel(season=season).plot(
        ax=ax, cmap='plasma',  vmin=0, vmax=0.1, add_colorbar=False
    )
    ax.set_title(f'{season.capitalize()} KD490 Trend')

# shared colorbar
fig.colorbar(
    plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(vmin=0, vmax=0.1)),
    ax=axes.ravel().tolist(), orientation='vertical', fraction=0.03, pad=0.04
)
plt.show()



In [ ]:
significant_w_seasons = seasonal_ds['pvalue'] < 0.05  # 95% confidence
trend_sig_seasons = seasonal_ds['slope'].where(significant_w_seasons)


In [ ]:
display(trend_sig_seasons)

In [ ]:
seasons = [ 'autumn','winter', 'spring', 'summer']
fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)

for ax, season in zip(axes.flat, seasons):
    trend_sig_seasons.sel(season=season).plot(
        ax=ax, cmap='RdBu_r', center=0, vmin=-0.002, vmax=0.002, add_colorbar=False
    )
    ax.set_title(f'{season.capitalize()} Significant KD490 Trend')

# shared colorbar
fig.colorbar(
    plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(vmin=-0.002, vmax=0.002)),
    ax=axes.ravel().tolist(), orientation='vertical', fraction=0.03, pad=0.04
)

fig.savefig('Sig_Seasonal_KD490.png')
plt.show()

# Now - Chlorophyll!

Steps to take:
1. Use new_batch_file() function to extract all of the chlorophyll data
2. Use glob to combine all the files into one list, then use xr.open_mfdataset() to combine into one DataSet
3.

## File Extraction Starts here

make sure that the function new_batch_file(dataset,variables,region) has already been run, along with the username and password cells
Once downloaded, shouldn't have to run again. DONT run again.


**Should have made it so that the directory is also changable** I didn't though, so after running the below cell, you need to go into the file folder on the computer (local -> home -> joyvan -> notmessedup_directory and make a new folder ('chlorophyll'). then put all the downloads for chla_ireland_file into this folder. When combining all files, make sure to specify the chlorophyll file.

In [ ]:
## Source: North Atlantic Ocean Colour Plankton, Reflectance, Transparency and Optics MY L3 daily observations
## DOI (product):
## https://doi.org/10.48670/moi-00286
## Accessed: October 14, 2025

In [ ]:
chla_ireland_file = new_batch_file("cmems_obs-oc_atl_bgc-plankton_my_l3-multi-1km_P1D",["CHL"],[{"name":"ireland", "min_lon": -15.0, "min_lat": 48.0, "max_lon":-2.0, "max_lat": 59.0}])


In [ ]:
chla_file_list = sorted(glob.glob('/home/jovyan/notmessedup_directory/chlorophyll/*.nc'))

In [ ]:
chla_ds = xr.open_mfdataset(chla_file_list, concat_dim='time', combine='nested')

In [ ]:
chla_ds

### Exploratory Plots

In [ ]:
chla_ds['CHL'].sel(time='1998-08-15').T.plot.surface()
plt.show()


In [ ]:
chla_ds['CHL'].sel(time=slice("1997-12-01", "1998-02-28"),latitude=slice(51,56), longitude=slice(-12,-5.5)).mean(dim='time').plot.surface(cmap='plasma', vmin = 0,vmax = 20)
plt.gca().set_zlim(0,20)

In [ ]:
chla_ds['CHL'].sel(time=slice("1997-12-01", "1998-02-28"), latitude=slice(51,56), longitude=slice(-12,-5.5)).mean(dim='time').plot(cmap='YlGn',robust=True, levels=[0,1,2,3,4,5,6,10,15,20,40])
plt.show()

In [ ]:
chla_ds['CHL'].sel(time=slice("1998-06-01", "1999-08-28"), latitude=slice(51,56), longitude=slice(-12,-5.5)).mean(dim='time').plot(cmap='YlGn',robust=True, levels=[0,1,2,3,4,5,6,10,15,20,40])
plt.show()

In [ ]:
chla_ds['CHL'].sel(time=slice("1997-12-01", "1998-02-28"), ).mean(dim='time').plot(
    cmap='YlGn', robust=False,
    vmin = 0,vmax = 20)
plt.show()


## Weekly, Monthly, Seasonal DS

In [ ]:
weekly_chla = chla_ds['CHL'].resample(time='1W').mean()

In [ ]:
monthly_chla = chla_ds['CHL'].resample(time='1ME').mean()

In [ ]:
seasonal_chla = chla_ds['CHL'].resample(time='QS-DEC').mean()
seasonal_chla['season'] = seasonal_chla['time'].dt.season
seasonal_chla

## Statistics!

### Weekly

In [ ]:
## Need to run this with relevant dataframe prior to running any code with the function 'fit_with_uncertainty' in it. It provides the x.
t = weekly_chla['time']
time_years = t.dt.year + (t - pd.to_datetime(t.dt.year.astype(str) + '-01-01')) / (pd.to_datetime((t.dt.year + 1).astype(str) + '-01-01') - pd.to_datetime(t.dt.year.astype(str) + '-01-01'))

x = time_years.values.astype(float)

In [ ]:

slope, intercept, rvalue, pvalue, slope_stderr, intercept_stderr = xr.apply_ufunc(
    fit_with_uncertainty,
    weekly_chla,
    input_core_dims=[['time']],
    output_core_dims=[[], [], [], [], [], []],
    vectorize=True,
    dask='parallelized',
    dask_gufunc_kwargs={'allow_rechunk': True},
    output_dtypes=[float, float, float, float, float, float],
    kwargs={'x': x}
)


In [ ]:
weekly_trend_chla = xr.Dataset({
    'slope': slope,
    'intercept': intercept,
    'rvalue': rvalue,
    'pvalue': pvalue,
    'slope_stderr': slope_stderr,
    'intercept_stderr': intercept_stderr,
})

In [ ]:

weekly_trend_chla = weekly_trend_chla.compute()

In [ ]:

weekly_trend_chla.to_netcdf('Chl_a_trend_with_stats.nc')

In [ ]:
weekly_trend_chla = xr.open_dataset('thesis_data/Chl_a_trend_with_stats.nc')
weekly_trend_chla

In [ ]:
w_significant_chla = weekly_trend_chla['pvalue'] < 0.05  # 95% confidence
w_trend_sig_chla = weekly_trend_chla['slope'].where(w_significant_chla)


#### Plots

In [ ]:
w_trend_sig_chla.plot(
    cmap='RdBu_r',
    center=0,
    robust=True,
    cbar_kwargs={'label': 'Significant Chlorophyll Change (mg m^-3 per year)'}
)
plt.savefig('w_Significant_Trend_Chl_a.png')
plt.show()

In [ ]:
weekly_trend_chla['slope'].plot(
    cmap='RdBu_r',
    center=0,
    robust=True,
    cbar_kwargs={'label': 'Significant Chlorophyll Change (mg m^-3 per year)'}
)
plt.savefig('Weekly_Trend_Chl_a.png')
plt.show()

### Seasonal

In [ ]:

def assign_season(month):
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    elif month in [9, 10, 11]:
        return 'autumn'

season_labels = seasonal_chla['time'].dt.month.to_series().apply(assign_season)
seasonal_chla = seasonal_chla.assign_coords(season=('time', season_labels.values))


In [ ]:

seasonal_trends = {}

for season in ['winter', 'spring', 'summer', 'autumn']:
    season_data = seasonal_chla.where(seasonal_chla['season'] == season, drop=True)

    # Compute decimal years for regression
    t = season_data['time']
    time_years = t.dt.year + (t - pd.to_datetime(t.dt.year.astype(str) + '-01-01')) / \
                 (pd.to_datetime((t.dt.year + 1).astype(str) + '-01-01') - pd.to_datetime(t.dt.year.astype(str) + '-01-01'))
    x_season = time_years.values.astype(float)

    # Apply linear regression with uncertainty
    slope, intercept, rvalue, pvalue, slope_stderr, intercept_stderr = xr.apply_ufunc(
        fit_with_uncertainty,
        season_data,
        input_core_dims=[['time']],
        output_core_dims=[[], [], [], [], [], []],
        vectorize=True,
        dask='parallelized',
        dask_gufunc_kwargs={'allow_rechunk': True},
        output_dtypes=[float, float, float, float, float, float],
        kwargs={'x': x_season}
    )

    # Store per-season results
    seasonal_trends[season] = xr.Dataset({
        'slope': slope,
        'intercept': intercept,
        'rvalue': rvalue,
        'pvalue': pvalue,
        'slope_stderr': slope_stderr,
        'intercept_stderr': intercept_stderr
    })


In [ ]:

season_ds_list = []
for season, ds in seasonal_trends.items():
    ds = ds.expand_dims({'season': [season]})
    season_ds_list.append(ds)

seasonal_trend_chla = xr.concat(season_ds_list, dim='season')


In [ ]:

seasonal_trend_chla = seasonal_trend_chla.compute()

In [ ]:
encoding = {v: {"zlib": True, "complevel": 4} for v in seasonal_trend_chla.data_vars}
seasonal_trend_chla.to_netcdf('Chl_a_seasonal_trends.nc', encoding=encoding)

In [ ]:
seasonal_trend_chla = xr.open_dataset('thesis_data/Chl_a_seasonal_trends.nc')
seasonal_trend_chla


In [ ]:
sea_significant_chla = seasonal_trend_chla['pvalue'] < 0.05  # 95% confidence
sea_trend_sig_chla =seasonal_trend_chla['slope'].where(sea_significant_chla)


#### Plots

In [ ]:
slope_data = seasonal_trend_chla['slope'].values
vmin_val, vmax_val = np.nanpercentile(slope_data, [2, 98])
limit = max(abs(vmin_val),abs(vmax_val))
limit
## use this method to set the colorbar range

In [ ]:
seasons = [ 'autumn','winter', 'spring', 'summer']
fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)

slope_data = seasonal_trend_chla['slope'].values
vmin_val, vmax_val = np.nanpercentile(slope_data, [2, 98])
limit = max(abs(vmin_val), abs(vmax_val))

cmap = plt.colormaps.get_cmap('RdBu_r').copy()
cmap.set_bad(color='#EAEAF2')

for ax, season in zip(axes.flat, seasons):
    seasonal_trend_chla['slope'].sel(season=season).plot(
        ax=ax, cmap=cmap, center=0,vmin=-limit, vmax=limit, add_colorbar=False
    )
    ax.set_title(f'{season.capitalize()} Chlorophyll-a Trend')

# shared colorbar
fig.colorbar(
    plt.cm.ScalarMappable(cmap=cmap,norm=plt.Normalize(vmin=-limit, vmax=limit)),
    ax=axes.ravel().tolist(), orientation='vertical', fraction=0.03, pad=0.04
)

fig.savefig('Seasonal_Chl_a_trend.png')
plt.show()

In [ ]:
seasonal_avg_chla = {}
for season in seasons:
    seasonal_avg_chla[season] = (
        seasonal_chla.where(seasonal_chla['season'] == season, drop=True)
        .mean(dim='time', skipna=True)
        .compute()
    )


fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)

data = seasonal_avg_chla['summer'].values
vmax_val = np.nanpercentile(data, 99)
vmin_val = 0

cmap = plt.colormaps.get_cmap('YlGn').copy()
cmap.set_bad(color='#EAEAF2')

for ax, season in zip(axes.flat, seasons):
    seasonal_avg_chla[season].plot(
        ax=ax, cmap=cmap,vmin=vmin_val, vmax=vmax_val, add_colorbar=False
    )
    ax.set_title(f'{season.capitalize()} Chlorophyll-a Average')

# shared colorbar
fig.colorbar(
    plt.cm.ScalarMappable(cmap=cmap,norm=plt.Normalize(vmin=vmin_val, vmax=vmax_val)),
    ax=axes.ravel().tolist(), orientation='vertical', fraction=0.03, pad=0.04
)

fig.savefig('Seasonal_Chl_a_Avg.png')
plt.show()

# Now - CDOM!

# Data download - Use this function (new_batch_file_with_dir)

In [4]:
os.environ['COPERNICUS_USERNAME'] = 'hdoran'
os.environ['COPERNICUS_PASSWORD'] = "ihateAngela69**##"

username = os.getenv('COPERNICUS_USERNAME')
password = os.getenv('COPERNICUS_PASSWORD')
## messed up! should've done at least 56 degrees north, or maybe even 57
## 55 deg N cuts into the northern end of the island
def new_batch_file_with_dir(dataset_id, variables, regions, directory):
    start_date = datetime(1997, 9, 4)
    end_date = datetime(2025, 9, 4)

    results = {}
    years = range(start_date.year, end_date.year + 1)

    for region in regions:
        region_name = region["name"]
        region_file_paths = []

        for year in years:
            year_start = datetime(year, 1, 1)
            year_end = datetime(year, 12, 31)
            if year == start_date.year:
                year_start = start_date
            if year == end_date.year:
                year_end = end_date

            # Download the data for this specific year
            result = copernicusmarine.subset(
                dataset_id=dataset_id,
                variables=variables,
                minimum_longitude=region["min_lon"],
                maximum_longitude=region["max_lon"],
                minimum_latitude=region["min_lat"],
                maximum_latitude=region["max_lat"],
                start_datetime=year_start.strftime('%Y-%m-%dT%H:%M:%S'),
                end_datetime=year_end.strftime('%Y-%m-%dT%H:%M:%S'),
                output_directory='/home/jovyan/notmessedup_directory'+directory,
                username=username,
                password=password
            )

            # Save only the file path (not the dataset object)
            region_file_paths.append(result.file_path)

            # Free memory
            del result
            gc.collect()

        results[region_name] = region_file_paths

    return results


In [ ]:
## Source: North Atlantic Ocean Colour Plankton, Reflectance, Transparency and Optics MY L3 daily observations
## DOI (product):
## https://doi.org/10.48670/moi-00286
## Accessed: October 16, 2025

In [ ]:
cdom_ireland_file = new_batch_file_with_dir("cmems_obs-oc_atl_bgc-optics_my_l3-multi-1km_P1D",["CDM"],[{"name":"ireland", "min_lon": -15.0, "min_lat": 48.0, "max_lon":-2.0, "max_lat": 59.0}],'/CDOM')

## When Starting With Daily Data - Start here to compile all netCDF files for CDOM

In [ ]:
cdom_file_list = sorted(glob.glob('/home/jovyan/notmessedup_directory/CDOM/*.nc'))
cdom_ds = xr.open_mfdataset(cdom_file_list, concat_dim='time', combine='nested')
display(cdom_ds)

In [ ]:
cdom_ds['CDM'].sel(time='2002-08-02').plot()

## Weekly, Monthly, Seasonal DS

In [ ]:
weekly_cdom = cdom_ds['CDM'].resample(time='1W').mean()
monthly_cdom = cdom_ds['CDM'].resample(time='1ME').mean()
seasonal_cdom = cdom_ds['CDM'].resample(time='QS-DEC').mean()
seasonal_cdom['season'] = seasonal_cdom['time'].dt.season
seasonal_cdom

## Statistics!

### Weekly

In [ ]:
## Need to run this with relevant dataframe prior to running any code with the function 'fit_with_uncertainty' in it. It provides the x.
t = weekly_cdom['time']
time_years = t.dt.year + (t - pd.to_datetime(t.dt.year.astype(str) + '-01-01')) / (
            pd.to_datetime((t.dt.year + 1).astype(str) + '-01-01') - pd.to_datetime(t.dt.year.astype(str) + '-01-01'))

x = time_years.values.astype(float)

In [ ]:
### Need to go up to the KD490 Data and rerun fit_with_uncertianty function

In [ ]:

slope, intercept, rvalue, pvalue, slope_stderr, intercept_stderr = xr.apply_ufunc(
    fit_with_uncertainty,
    weekly_cdom,
    input_core_dims=[['time']],
    output_core_dims=[[], [], [], [], [], []],
    vectorize=True,
    dask='parallelized',
    dask_gufunc_kwargs={'allow_rechunk': True},
    output_dtypes=[float, float, float, float, float, float],
    kwargs={'x': x}
)

In [ ]:

weekly_trend_cdom = xr.Dataset({
    'slope': slope,
    'intercept': intercept,
    'rvalue': rvalue,
    'pvalue': pvalue,
    'slope_stderr': slope_stderr,
    'intercept_stderr': intercept_stderr,
})

In [ ]:

weekly_trend_cdom = weekly_trend_cdom.compute()

In [ ]:

weekly_trend_cdom.to_netcdf('CDOM_trend_with_stats.nc')
weekly_trend_cdom = xr.open_dataset('thesis_data/CDOM_trend_with_stats.nc')
weekly_trend_cdom

In [ ]:
w_significant_cdom = weekly_trend_cdom['pvalue'] < 0.05  # 95% confidence
w_trend_sig_cdom = weekly_trend_cdom['slope'].where(w_significant_cdom)

### Plots

In [ ]:
weekly_trend_cdom['slope'].plot(
    cmap='RdBu_r',
    center=0,
    robust=True,
    cbar_kwargs={'label': 'CDOM Change (m^-1 per year)'}
)
plt.savefig('Weekly_Trend_CDOM.png')
plt.show()

In [ ]:
w_trend_sig_cdom.plot(
    cmap='RdBu_r',
    center=0,
    robust=True,
    cbar_kwargs={'label': 'Significant CDOM Change (m^-1 per year)'}
)
plt.savefig('Weekly_Trend_Sig_CDOM.png')
plt.show()

In [ ]:
w_trend_sig_cdom.sel(latitude=slice(51,56)).sel(longitude=slice(-11,-7)).plot(
    cmap='RdBu_r',
    center=0,
    robust=True,
    cbar_kwargs={'label': 'Significant CDOM Change (m^-1 per year)'}
)
plt.savefig('Weekly_Trend_Sig_CDOM_WestCoast.png')
plt.show()

### Seasonal

In [ ]:

def assign_season(month):
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    elif month in [9, 10, 11]:
        return 'autumn'


season_labels = seasonal_cdom['time'].dt.month.to_series().apply(assign_season)
seasonal_cdom = seasonal_cdom.assign_coords(season=('time', season_labels.values))

In [ ]:

seasonal_trends = {}

for season in ['winter', 'spring', 'summer', 'autumn']:
    season_data = seasonal_cdom.where(seasonal_cdom['season'] == season, drop=True)

    # Compute decimal years for regression
    t = season_data['time']
    time_years = t.dt.year + (t - pd.to_datetime(t.dt.year.astype(str) + '-01-01')) / \
                 (pd.to_datetime((t.dt.year + 1).astype(str) + '-01-01') - pd.to_datetime(
                     t.dt.year.astype(str) + '-01-01'))
    x_season = time_years.values.astype(float)

    # Apply linear regression with uncertainty
    slope, intercept, rvalue, pvalue, slope_stderr, intercept_stderr = xr.apply_ufunc(
        fit_with_uncertainty,
        season_data,
        input_core_dims=[['time']],
        output_core_dims=[[], [], [], [], [], []],
        vectorize=True,
        dask='parallelized',
        dask_gufunc_kwargs={'allow_rechunk': True},
        output_dtypes=[float, float, float, float, float, float],
        kwargs={'x': x_season}
    )

    # Store per-season results
    seasonal_trends[season] = xr.Dataset({
        'slope': slope,
        'intercept': intercept,
        'rvalue': rvalue,
        'pvalue': pvalue,
        'slope_stderr': slope_stderr,
        'intercept_stderr': intercept_stderr
    })

In [ ]:

season_ds_list = []
for season, ds in seasonal_trends.items():
    ds = ds.expand_dims({'season': [season]})
    season_ds_list.append(ds)

In [ ]:

seasonal_trend_cdom = xr.concat(season_ds_list, dim='season')

In [ ]:

seasonal_trend_cdom = seasonal_trend_cdom.compute()

In [ ]:
encoding = {v: {"zlib": True, "complevel": 4} for v in seasonal_trend_cdom.data_vars}
seasonal_trend_cdom.to_netcdf('CDOM_seasonal_trends.nc', encoding=encoding)
seasonal_trend_cdom = xr.open_dataset('thesis_data/CDOM_seasonal_trends.nc')
seasonal_trend_cdom

In [ ]:

sea_significant_cdom = seasonal_trend_cdom['pvalue'] < 0.05  # 95% confidence
sea_trend_sig_cdom = seasonal_trend_cdom['slope'].where(sea_significant_cdom)


### Plots

In [ ]:
seasons = [ 'autumn','winter', 'spring', 'summer']
fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)

slope_data = seasonal_trend_cdom['slope'].values
vmin_val, vmax_val = np.nanpercentile(slope_data, [2, 98])
limit = max(abs(vmin_val), abs(vmax_val))

cmap = plt.colormaps.get_cmap('RdBu_r').copy()
cmap.set_bad(color='#EAEAF2')

for ax, season in zip(axes.flat, seasons):
    seasonal_trend_cdom['slope'].sel(season=season).plot(
        ax=ax, cmap=cmap, center=0,vmin=-limit, vmax=limit, add_colorbar=False
    )
    ax.set_title(f'{season.capitalize()} CDOM Trend')

# shared colorbar
fig.colorbar(
    plt.cm.ScalarMappable(cmap=cmap,norm=plt.Normalize(vmin=-limit, vmax=limit)),
    ax=axes.ravel().tolist(), orientation='vertical', fraction=0.03, pad=0.04
)

fig.savefig('Seasonal_CDOM_trend.png')
plt.show()

In [ ]:
seasons = [ 'autumn','winter', 'spring', 'summer']
fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)

slope_data = sea_trend_sig_cdom.values
vmin_val, vmax_val = np.nanpercentile(slope_data, [2, 98])
limit = max(abs(vmin_val), abs(vmax_val))

cmap = plt.colormaps.get_cmap('RdBu_r').copy()
cmap.set_bad(color='#EAEAF2')

for ax, season in zip(axes.flat, seasons):
    sea_trend_sig_cdom.sel(season=season).plot(
        ax=ax, cmap=cmap, center=0,vmin=-limit, vmax=limit, add_colorbar=False
    )
    ax.set_title(f'{season.capitalize()} Significant CDOM Trend')

# shared colorbar
fig.colorbar(
    plt.cm.ScalarMappable(cmap=cmap,norm=plt.Normalize(vmin=-limit, vmax=limit)),
    ax=axes.ravel().tolist(), orientation='vertical', fraction=0.03, pad=0.04
)

fig.savefig('Seasonal_CDOM_trend_significant.png')
plt.show()

In [ ]:
seasonal_avg_cdom = {}
for season in seasons:
    seasonal_avg_cdom[season] = (
        seasonal_cdom.where(seasonal_cdom['season'] == season, drop=True)
        .mean(dim='time', skipna=True)
        .compute()
    )


fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)

data = seasonal_avg_cdom['summer'].values
vmax_val = np.nanpercentile(data, 99)
vmin_val = 0

cmap = plt.colormaps.get_cmap('YlGn').copy()
cmap.set_bad(color='#EAEAF2')

for ax, season in zip(axes.flat, seasons):
    seasonal_avg_cdom[season].plot(
        ax=ax, cmap=cmap,vmin=vmin_val, vmax=vmax_val, add_colorbar=False
    )
    ax.set_title(f'{season.capitalize()} CDOM Average')

# shared colorbar
fig.colorbar(
    plt.cm.ScalarMappable(cmap=cmap,norm=plt.Normalize(vmin=vmin_val, vmax=vmax_val)),
    ax=axes.ravel().tolist(), orientation='vertical', fraction=0.03, pad=0.04
)

fig.savefig('Seasonal_CDOM_Avg.png')
plt.show()

# Now - SST!

In [ ]:
## Different Source! Same Area
## Source: European North West Shelf/Iberia Biscay Irish Seas - High Resolution L4 Sea Surface Temperature Reprocessed
## DOI (product):
## https://doi.org/10.48670/moi-00153
## Date Accessed: October 20, 2025

In [6]:
sst_ireland_file = new_batch_file_with_dir("cmems-IFREMER-ATL-SST-L4-REP-OBS_FULL_TIME_SERIE",["analysed_sst"],[{"name":"ireland", "min_lon": -15.0, "min_lat": 48.0, "max_lon":-2.0, "max_lat": 59.0}],'/SST2')

INFO - 2026-07-05T16:30:11Z - Selected dataset version: "202411"
INFO - 2026-07-05T16:30:11Z - Selected dataset part: "default"
INFO - 2026-07-05T16:30:12Z - Starting download. Please wait...


  0%|          | 0/42 [00:00<?, ?it/s]

INFO - 2026-07-05T16:30:34Z - Successfully downloaded to \home\jovyan\notmessedup_directory\SST2\cmems-IFREMER-ATL-SST-L4-REP-OBS_FULL_TIME_SERIE_analysed_sst_14.97W-2.02W_48.02N-58.98N_1997-09-04-1997-12-31.nc
INFO - 2026-07-05T16:30:35Z - Selected dataset version: "202411"
INFO - 2026-07-05T16:30:35Z - Selected dataset part: "default"
INFO - 2026-07-05T16:30:36Z - Starting download. Please wait...


  0%|          | 0/130 [00:00<?, ?it/s]

INFO - 2026-07-05T16:31:15Z - Successfully downloaded to \home\jovyan\notmessedup_directory\SST2\cmems-IFREMER-ATL-SST-L4-REP-OBS_FULL_TIME_SERIE_analysed_sst_14.97W-2.02W_48.02N-58.98N_1998-01-01-1998-12-31.nc
INFO - 2026-07-05T16:31:18Z - Selected dataset version: "202411"
INFO - 2026-07-05T16:31:18Z - Selected dataset part: "default"
INFO - 2026-07-05T16:31:19Z - Starting download. Please wait...


...

INFO - 2026-07-05T16:57:05Z - Successfully downloaded to \home\jovyan\notmessedup_directory\SST2\cmems-IFREMER-ATL-SST-L4-REP-OBS_FULL_TIME_SERIE_analysed_sst_14.97W-2.02W_48.02N-58.98N_2023-01-01-2023-12-31.nc
INFO - 2026-07-05T16:57:06Z - Selected dataset version: "202411"
INFO - 2026-07-05T16:57:06Z - Selected dataset part: "default"
INFO - 2026-07-05T16:57:07Z - Starting download. Please wait...


  0%|          | 0/149 [00:00<?, ?it/s]

INFO - 2026-07-05T16:57:44Z - Successfully downloaded to \home\jovyan\notmessedup_directory\SST2\cmems-IFREMER-ATL-SST-L4-REP-OBS_FULL_TIME_SERIE_analysed_sst_14.97W-2.02W_48.02N-58.98N_2024-01-01-2024-12-31.nc
INFO - 2026-07-05T16:57:46Z - Selected dataset version: "202411"
INFO - 2026-07-05T16:57:46Z - Selected dataset part: "default"
INFO - 2026-07-05T16:57:48Z - Starting download. Please wait...


  0%|          | 0/88 [00:00<?, ?it/s]

INFO - 2026-07-05T16:58:19Z - Successfully downloaded to \home\jovyan\notmessedup_directory\SST2\cmems-IFREMER-ATL-SST-L4-REP-OBS_FULL_TIME_SERIE_analysed_sst_14.97W-2.02W_48.02N-58.98N_2025-01-01-2025-09-04.nc


In [ ]:
## All files available to be downloaded at this time were downloaded. Only goes until Dec 2023.

In [4]:
sst_file_list = sorted(glob.glob('/home/jovyan/notmessedup_directory/SST2/*.nc'))
sst_ds = xr.open_mfdataset(sst_file_list, concat_dim='time', combine='nested')
display(sst_ds)

<xarray.Dataset> Size: 5GB
Dimensions:       (time: 10228, latitude: 220, longitude: 260)
Coordinates:
  * time          (time) datetime64[ns] 82kB 1997-09-04 ... 2025-09-04
  * latitude      (latitude) float64 2kB 48.02 48.08 48.12 ... 58.88 58.92 58.98
  * longitude     (longitude) float64 2kB -14.97 -14.93 -14.88 ... -2.075 -2.025
Data variables:
    analysed_sst  (time, latitude, longitude) float64 5GB dask.array<chunksize=(119, 220, 260), meta=np.ndarray>
Attributes:
    Conventions:               CF-1.7,ACDD-1.3,ISO 8601
    comment:                   WARNING:Some applications are unable to proper...
    source:                    Odyssea-RepL4 Processor
    history:                   file originally produced by Ifremer/Cersat wit...
    title:                     Merged collation of sea surface temperature fr...
    references:                Product User Manual for Level 4 ODYSSEA Sea Su...
    institution:               Institut Francais de Recherche pour l Exploita...
    contact:                   jean.francois.piolle@ifremer.fr;emmanuelle.aut...
    copernicusmarine_version:  2.2.2

## Weekly, Monthly, Seasonal

In [5]:
weekly_sst = sst_ds['analysed_sst'].resample(time='1W').mean()
monthly_sst = sst_ds['analysed_sst'].resample(time='1ME').mean()
seasonal_sst = sst_ds['analysed_sst'].resample(time='QS-DEC').mean()
seasonal_sst['season'] = seasonal_sst['time'].dt.season

## Statistics

### Weekly

In [6]:
## Need to run this with relevant dataframe prior to running any code with the function 'fit_with_uncertainty' in it. It provides the x.
t = weekly_sst['time']
time_years = t.dt.year + (t - pd.to_datetime(t.dt.year.astype(str) + '-01-01')) / (
        pd.to_datetime((t.dt.year + 1).astype(str) + '-01-01') - pd.to_datetime(t.dt.year.astype(str) + '-01-01'))

x = time_years.values.astype(float)

In [7]:
### Need to go up to the KD490 Data and rerun fit_with_uncertianty function

slope, intercept, rvalue, pvalue, slope_stderr, intercept_stderr = xr.apply_ufunc(
    fit_with_uncertainty,
    weekly_sst,
    input_core_dims=[['time']],
    output_core_dims=[[], [], [], [], [], []],
    vectorize=True,
    dask='parallelized',
    dask_gufunc_kwargs={'allow_rechunk': True},
    output_dtypes=[float, float, float, float, float, float],
    kwargs={'x': x}
)

In [8]:

weekly_trend_sst = xr.Dataset({
    'slope': slope,
    'intercept': intercept,
    'rvalue': rvalue,
    'pvalue': pvalue,
    'slope_stderr': slope_stderr,
    'intercept_stderr': intercept_stderr,
})

In [9]:
weekly_trend_sst

<xarray.Dataset> Size: 3MB
Dimensions:           (latitude: 220, longitude: 260)
Coordinates:
  * latitude          (latitude) float64 2kB 48.02 48.08 48.12 ... 58.92 58.98
  * longitude         (longitude) float64 2kB -14.97 -14.93 ... -2.075 -2.025
Data variables:
    slope             (latitude, longitude) float64 458kB dask.array<chunksize=(6, 7), meta=np.ndarray>
    intercept         (latitude, longitude) float64 458kB dask.array<chunksize=(6, 7), meta=np.ndarray>
    rvalue            (latitude, longitude) float64 458kB dask.array<chunksize=(6, 7), meta=np.ndarray>
    pvalue            (latitude, longitude) float64 458kB dask.array<chunksize=(6, 7), meta=np.ndarray>
    slope_stderr      (latitude, longitude) float64 458kB dask.array<chunksize=(6, 7), meta=np.ndarray>
    intercept_stderr  (latitude, longitude) float64 458kB dask.array<chunksize=(6, 7), meta=np.ndarray>

In [12]:

#weekly_trend_sst = weekly_trend_sst.compute()

MemoryError: 

In [10]:

weekly_trend_sst.to_netcdf('thesis_data/SST_trend_with_stats.nc', engine='h5netcdf')

In [13]:
weekly_trend_sst = xr.open_dataset('thesis_data/SST_trend_with_stats.nc')
weekly_trend_sst

MemoryError: 

In [ ]:
w_significant_sst = weekly_trend_sst['pvalue'] < 0.05  # 95% confidence
w_trend_sig_sst = weekly_trend_sst['slope'].where(w_significant_sst)

In [ ]:
weekly_trend_sst['slope'].plot(
    cmap='RdBu_r',
    center=0,
    robust=True,
    cbar_kwargs={'label': 'SST Change (deg K per year)'}
)
plt.savefig('Weekly_Trend_SST.png')
plt.show()

In [ ]:
## Should look at overall temperature change for the region, then look at regional temperature changes (line plot for regions - NW vs S/SE)

In [ ]:
sst_ds = sst_ds.compute()

In [ ]:
sst_ds

In [ ]:
sst_da_c = sst_ds.analysed_sst - 273.15
sst_ds_c = sst_da_c.to_dataset()

In [ ]:
sst_ds_c = sst_ds_c.assign_attrs(units='Celsius', description = 'Original Kelvin data converted to Deg Celsius')

In [ ]:
sst_ds_c

### Plots!

In [ ]:
sst_ds['analysed_sst'].mean(dim='time').plot(
    robust=True,
    cbar_kwargs={'label': 'SST (deg K)'}
)
plt.savefig('SST.png')
plt.show()

In [ ]:
sst_ds['analysed_sst'].isel(time = slice(1997-9-4,1999-9-4)).mean(dim='time').plot(
    robust=True,
    cbar_kwargs={'label': 'SST (deg K)'}
)
plt.savefig('SST.png')
plt.show()

In [ ]:
sst_ds['analysed_sst'].isel(time = slice(2020-9-4,2022-9-4)).mean(dim='time').plot(
    robust=True,
    cbar_kwargs={'label': 'SST (deg K)'}
)
plt.savefig('SST.png')
plt.show()

In [ ]:
sst_ds['analysed_sst'].mean(dim=['latitude','longitude']).plot()

In [ ]:
sst_ds['analysed_sst'].sel(latitude=slice(54,58),longitude=slice(-15,-9)).mean(dim=['latitude','longitude']).plot()
plt.show()

In [ ]:
t = sst_ds['time']
time_years = t.dt.year + (t - pd.to_datetime(t.dt.year.astype(str) + '-01-01')) / (
        pd.to_datetime((t.dt.year + 1).astype(str) + '-01-01') - pd.to_datetime(t.dt.year.astype(str) + '-01-01'))

x = time_years.values.astype(float)

In [ ]:
northwest_ire = sst_ds['analysed_sst'].sel(latitude=slice(54,58),longitude=slice(-15,-9)).mean(dim=['latitude','longitude'])
south_ire = sst_ds['analysed_sst'].sel(latitude=slice(48,52)).mean(dim=['latitude','longitude'])

coeffs_nw = np.polyfit(x, northwest_ire.values, 1)
coeffs_s  = np.polyfit(x, south_ire.values, 1)

# Evaluate the trend lines
trend_nw = np.polyval(coeffs_nw, x)
trend_s  = np.polyval(coeffs_s, x)



In [ ]:
plt.figure(figsize = (15,8))

plt.plot(northwest_ire['time'], northwest_ire, label='Northwest of Ireland', color='blue')
plt.plot(south_ire['time'], south_ire, label='South of Ireland', color='red')

plt.plot(northwest_ire['time'], trend_nw, label='Northwest trend', color='blue',linestyle='dashed')
plt.plot(south_ire['time'], trend_s, label='Southern trend', color='red',linestyle='dashed')

plt.text(northwest_ire['time'][0].values, trend_nw[0]-3, f"{trend_nw[0]:.2f}°C", color='blue', ha='left', va='bottom')
plt.text(northwest_ire['time'][-1].values, trend_nw[-1]-3, f"{trend_nw[-1]:.2f}°C", color='blue', ha='left', va='bottom' )

plt.text(south_ire['time'][0].values, trend_s[0]+5, f"{trend_s[0]:.2f}°C", color='red', ha='left', va='bottom' )
plt.text(south_ire['time'][-1].values, trend_s[-1]+5, f"{trend_s[-1]:.2f}°C", color='red', ha='left', va='bottom'  )

plt.xlabel('Time')
plt.ylabel('Degree K')
plt.legend()
plt.title('Sea Surface Temperature of Ireland - Regional Differences')
plt.plot()

Changed data to deg C

In [ ]:
t = sst_ds_c['time']
time_years = t.dt.year + (t - pd.to_datetime(t.dt.year.astype(str) + '-01-01')) / (
        pd.to_datetime((t.dt.year + 1).astype(str) + '-01-01') - pd.to_datetime(t.dt.year.astype(str) + '-01-01'))

x = time_years.values.astype(float)
northwest_ire_c = sst_ds_c['analysed_sst'].sel(latitude=slice(54, 58), longitude=slice(-15, -9)).mean( dim=['latitude', 'longitude'])
south_ire_c = sst_ds_c['analysed_sst'].sel(latitude=slice(48, 52)).mean(dim=['latitude', 'longitude'])

coeffs_nw_c = np.polyfit(x, northwest_ire_c.values, 1)
coeffs_s_c = np.polyfit(x, south_ire_c.values, 1)

# Evaluate the trend lines
trend_nw_c = np.polyval(coeffs_nw_c, x)
trend_s_c = np.polyval(coeffs_s_c, x)

In [ ]:

plt.figure(figsize=(15, 8))

plt.plot(northwest_ire_c['time'], northwest_ire_c, label='Northwest of Ireland', color='blue')
plt.plot(south_ire_c['time'], south_ire_c, label='South of Ireland', color='red')

plt.plot(northwest_ire_c['time'], trend_nw_c, label='Northwest trend', color='blue', linestyle='dashed')
plt.plot(south_ire_c['time'], trend_s_c, label='Southern trend', color='red', linestyle='dashed')

plt.text(northwest_ire_c['time'][0].values, trend_nw_c[0] - 3, f"{trend_nw_c[0]:.2f}°C", color='blue', ha='left', va='bottom')
plt.text(northwest_ire_c['time'][-1].values, trend_nw_c[-1] - 3, f"{trend_nw_c[-1]:.2f}°C", color='blue', ha='left',
         va='bottom')

plt.text(south_ire_c['time'][0].values, trend_s_c[0] + 5, f"{trend_s_c[0]:.2f}°C", color='red', ha='left', va='bottom')
plt.text(south_ire_c['time'][-1].values, trend_s_c[-1] + 5, f"{trend_s_c[-1]:.2f}°C", color='red', ha='left', va='bottom')

plt.xlabel('Time')
plt.ylabel('Degree C')
plt.legend()
plt.title('Sea Surface Temperature of Ireland - Regional Differences')
plt.plot()

# SPM!

In [ ]:
## Source: North Atlantic Ocean Colour Plankton, Reflectance, Transparency and Optics MY L3 daily observations
## DOI (product):
## https://doi.org/10.48670/moi-00286
## Accessed: October 28, 2025

In [ ]:
spm_ireland_file = new_batch_file_with_dir("cmems_obs-oc_atl_bgc-transp_my_l3-multi-1km_P1D",["SPM"],[{"name":"ireland", "min_lon": -15.0, "min_lat": 48.0, "max_lon":-2.0, "max_lat": 59.0}],'/SPM')

In [ ]:
spm_file_list = sorted(glob.glob('/home/jovyan/notmessedup_directory/SPM/*.nc'))
spm_ds = xr.open_mfdataset(spm_file_list, concat_dim='time', combine='nested')
display(spm_ds)

In [ ]:
weekly_spm = spm_ds['SPM'].resample(time='1W').mean()
monthly_spm = spm_ds['SPM'].resample(time='1ME').mean()
seasonal_spm = spm_ds['SPM'].resample(time='QS-DEC').mean()
seasonal_spm['season'] = seasonal_spm['time'].dt.season
seasonal_spm

## Statistics!

### Weekly

In [ ]:
## Need to run this with relevant dataframe prior to running any code with the function 'fit_with_uncertainty' in it. It provides the x.

In [ ]:
t = weekly_spm['time']
time_years = t.dt.year + (t - pd.to_datetime(t.dt.year.astype(str) + '-01-01')) / (
        pd.to_datetime((t.dt.year + 1).astype(str) + '-01-01') - pd.to_datetime(t.dt.year.astype(str) + '-01-01'))

x = time_years.values.astype(float)

In [ ]:
### Need to go up to the KD490 Data and rerun fit_with_uncertianty function

slope, intercept, rvalue, pvalue, slope_stderr, intercept_stderr = xr.apply_ufunc(
    fit_with_uncertainty,
    weekly_spm,
    input_core_dims=[['time']],
    output_core_dims=[[], [], [], [], [], []],
    vectorize=True,
    dask='parallelized',
    dask_gufunc_kwargs={'allow_rechunk': True},
    output_dtypes=[float, float, float, float, float, float],
    kwargs={'x': x}
)

In [ ]:

weekly_trend_spm = xr.Dataset({
    'slope': slope,
    'intercept': intercept,
    'rvalue': rvalue,
    'pvalue': pvalue,
    'slope_stderr': slope_stderr,
    'intercept_stderr': intercept_stderr,
})

In [ ]:

weekly_trend_spm = weekly_trend_spm.compute()

In [ ]:

weekly_trend_spm.to_netcdf('SPM_trend_with_stats.nc')
weekly_trend_spm = xr.open_dataset('thesis_data/SPM_trend_with_stats.nc')
weekly_trend_spm

In [ ]:
w_significant_spm = weekly_trend_spm['pvalue'] < 0.05  # 95% confidence
w_trend_sig_spm = weekly_trend_spm['slope'].where(w_significant_spm)

### Plots

In [ ]:
cmap = plt.get_cmap('RdBu_r').copy()  # copy to avoid altering global cmap
cmap.set_bad(color='#EAEAF2')

weekly_trend_spm['slope'].plot(
    cmap=cmap,
    center=0,
    robust=True,
    cbar_kwargs={'label': 'SPM Change (g/m^3 per year)'}
)

plt.savefig('Weekly_Trend_SPM.png', dpi=600, bbox_inches='tight')
plt.show()

### Seasonal Trends

In [ ]:

def assign_season(month):
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    elif month in [9, 10, 11]:
        return 'autumn'


season_labels = seasonal_spm['time'].dt.month.to_series().apply(assign_season)
seasonal_spm = seasonal_spm.assign_coords(season=('time', season_labels.values))

In [ ]:

seasonal_trends = {}

for season in ['winter', 'spring', 'summer', 'autumn']:
    season_data = seasonal_spm.where(seasonal_spm['season'] == season, drop=True)

    # Compute decimal years for regression
    t = season_data['time']
    time_years = t.dt.year + (t - pd.to_datetime(t.dt.year.astype(str) + '-01-01')) / \
                 (pd.to_datetime((t.dt.year + 1).astype(str) + '-01-01') - pd.to_datetime(
                     t.dt.year.astype(str) + '-01-01'))
    x_season = time_years.values.astype(float)

    # Apply linear regression with uncertainty
    slope, intercept, rvalue, pvalue, slope_stderr, intercept_stderr = xr.apply_ufunc(
        fit_with_uncertainty,
        season_data,
        input_core_dims=[['time']],
        output_core_dims=[[], [], [], [], [], []],
        vectorize=True,
        dask='parallelized',
        dask_gufunc_kwargs={'allow_rechunk': True},
        output_dtypes=[float, float, float, float, float, float],
        kwargs={'x': x_season}
    )

    # Store per-season results
    seasonal_trends[season] = xr.Dataset({
        'slope': slope,
        'intercept': intercept,
        'rvalue': rvalue,
        'pvalue': pvalue,
        'slope_stderr': slope_stderr,
        'intercept_stderr': intercept_stderr
    })

season_ds_list = []
for season, ds in seasonal_trends.items():
    ds = ds.expand_dims({'season': [season]})
    season_ds_list.append(ds)

seasonal_trend_spm = xr.concat(season_ds_list, dim='season')

In [ ]:

seasonal_trend_spm = seasonal_trend_spm.compute()

In [ ]:

encoding = {v: {"zlib": True, "complevel": 4} for v in seasonal_trend_spm.data_vars}
seasonal_trend_spm.to_netcdf('SPM_seasonal_trends.nc', encoding=encoding)
seasonal_trend_spm = xr.open_dataset('thesis_data/SPM_seasonal_trends.nc')
seasonal_trend_spm

In [ ]:

sea_significant_spm = seasonal_trend_spm['pvalue'] < 0.05  # 95% confidence
sea_trend_sig_spm = seasonal_trend_spm['slope'].where(sea_significant_spm)

### Plots

In [ ]:
seasons = ['autumn', 'winter', 'spring', 'summer']
fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)

slope_data = seasonal_trend_spm['slope'].values
vmin_val, vmax_val = np.nanpercentile(slope_data, [2, 98])
limit = max(abs(vmin_val), abs(vmax_val))

cmap = plt.colormaps.get_cmap('RdBu_r').copy()
cmap.set_bad(color='#EAEAF2')

for ax, season in zip(axes.flat, seasons):
    seasonal_trend_spm['slope'].sel(season=season).plot(
        ax=ax, cmap=cmap, center=0, vmin=-limit, vmax=limit, add_colorbar=False
    )
    ax.set_title(f'{season.capitalize()} SPM Trend')

# shared colorbar
fig.colorbar(
    plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=-limit, vmax=limit)),
    ax=axes.ravel().tolist(), orientation='vertical', fraction=0.03, pad=0.04
)

fig.savefig('Seasonal_SPM_trend.png', dpi=300, bbox_inches='tight')
plt.show()